# Netflix Movies and TV Shows: Data Cleaning

In this notebook I load the [Netflix Movies and TV Shows dataset](https://www.kaggle.com/datasets/shivamb/netflix-shows) from Kaggle and clean it with Pandas.

The goal is not just to drop nulls. For each column I look at **why** values are missing and make a deliberate decision about how to handle it.

**Steps**
1. Download and load the data
2. Inspect it (`.head()`, `.info()`, `.describe()`)
3. Find and handle missing values column by column
4. Fix mixed-type columns (`duration`)
5. Parse dates into `datetime`
6. Export the cleaned data to a new CSV

## 1. Download and load the data

In [11]:
%pip install kagglehub pandas


Defaulting to user installation because normal site-packages is not writeable

  Attempting uninstall: certifi

    Found existing installation: certifi 2026.1.4

    Uninstalling certifi-2026.1.4:

      Successfully uninstalled certifi-2026.1.4

   ---------------------------------------- 0/3 [certifi]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -----------------


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\akang\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import os

import kagglehub
import pandas as pd

path = kagglehub.dataset_download("shivamb/netflix-shows")
print("Path to dataset files:", path)

df = pd.read_csv(os.path.join(path, "netflix_titles.csv"))
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")

C:\Users\akang\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.34M/1.34M [00:01<00:00, 972kB/s]

Extracting files...


Path to dataset files: C:\Users\akang\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5
Loaded 8,807 rows and 12 columns


## 2. Inspect the DataFrame

In [13]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [15]:
# Numeric columns
df.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [16]:
# Text columns: counts, number of unique values, most common value
df.describe(include="object")

,show_id,type,title,director,cast,country,date_added,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,17,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,3207,1793,362,4


**What I notice**
- Several text columns have fewer non-null values than the total row count, so there are missing values.
- `date_added` is stored as text, not as a date.
- `duration` is text such as `"90 min"` or `"2 Seasons"`, which mixes two different units in one column.

## 3. Missing values

In [17]:
missing = df.isna().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(df) * 100).round(2)
missing[missing["missing"] > 0].sort_values("missing", ascending=False)

,missing,percent
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03


In [18]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate show_id values:", df["show_id"].duplicated().sum())

Duplicate rows: 0
Duplicate show_id values: 0


### Why are values missing, and what should I do about each column?

| Column | Likely reason | Decision |
|---|---|---|
| `director` | Many documentaries, stand-up specials and TV shows have no single credited director | Fill with `"Unknown"` and keep the rows. Dropping them would remove a large share of the catalogue. |
| `cast` | Documentaries and some reality shows have no listed cast | Fill with `"Unknown"` |
| `country` | Production country was not recorded for some titles | Fill with `"Unknown"` |
| `date_added` | A handful of titles have no recorded date | Leave as missing (`NaT`). Inventing a date would add false information. |
| `rating` | A few rows are blank, and a few have the **duration typed into the rating column** by mistake | Move misplaced durations to `duration`, then fill the rest with `"Unknown"` |
| `duration` | Missing only where it was entered in the `rating` column | Recovered from `rating` (see step 4) |

### Prepare: clean whitespace and remove duplicates

In [19]:
clean = df.copy()

# Strip stray spaces from every text column; treat empty strings as missing
text_cols = clean.select_dtypes(include="object").columns
for col in text_cols:
    clean[col] = clean[col].str.strip()
clean[text_cols] = clean[text_cols].replace("", pd.NA)

before = len(clean)
clean = clean.drop_duplicates()
print(f"Removed {before - len(clean)} duplicate rows")

Removed 0 duplicate rows


### Recover durations that were typed into the `rating` column

In [21]:
misplaced = clean["rating"].str.contains("min", case=False, na=False)
print("Rows with a duration in the rating column:", misplaced.sum())
clean.loc[misplaced, ["title", "rating", "duration"]]

Rows with a duration in the rating column: 3


,title,rating,duration
5541,Louis C.K. 2017,74 min,NaN
5794,Louis C.K.: Hilarious,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [22]:
fix = misplaced & clean["duration"].isna()
clean.loc[fix, "duration"] = clean.loc[fix, "rating"]
clean.loc[misplaced, "rating"] = pd.NA

### Fill the remaining missing values

In [23]:
for col in ["director", "cast", "country", "rating"]:
    clean[col] = clean[col].fillna("Unknown")

clean.isna().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64

## 4. Fix the mixed-type `duration` column

`duration` mixes minutes (`"90 min"`, used by movies) and seasons (`"2 Seasons"`, used by TV shows). I split it into two numeric columns so each can be analysed properly.

In [24]:
parts = clean["duration"].str.extract(r"(?P<value>\d+)\s*(?P<unit>[A-Za-z]+)")
value = pd.to_numeric(parts["value"]).astype("Int64")
unit = parts["unit"].str.lower()

clean["duration_minutes"] = value.where(unit.eq("min"))
clean["seasons"] = value.where(unit.str.startswith("season", na=False))

# The original text column is now redundant
clean = clean.drop(columns="duration")

clean[["type", "title", "duration_minutes", "seasons"]].head()

,type,title,duration_minutes,seasons
0,Movie,Dick Johnson Is Dead,90,<NA>
1,TV Show,Blood & Water,<NA>,2
2,TV Show,Ganglands,<NA>,1
3,TV Show,Jailbirds New Orleans,<NA>,1
4,TV Show,Kota Factory,<NA>,2


In [25]:
# Sanity check: movies should have minutes, TV shows should have seasons
clean.groupby("type", observed=True)[["duration_minutes", "seasons"]].count()

,duration_minutes,seasons
type,,
Movie,6131,0
TV Show,0,2676


## 5. Parse date columns

`date_added` looks like `"August 14, 2020"`. I convert it to `datetime` and add year and month columns, which are handy for analysis.

In [26]:
clean["date_added"] = pd.to_datetime(
    clean["date_added"], format="%B %d, %Y", errors="coerce"
)

clean["year_added"] = clean["date_added"].dt.year.astype("Int64")
clean["month_added"] = clean["date_added"].dt.month.astype("Int64")

print("Dates that could not be parsed or were missing:", clean["date_added"].isna().sum())

Dates that could not be parsed or were missing: 10


### Use sensible data types

In [27]:
clean["type"] = clean["type"].astype("category")
clean["rating"] = clean["rating"].astype("category")

## 6. Check the result

In [28]:
clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   show_id           8807 non-null   object        
 1   type              8807 non-null   category      
 2   title             8807 non-null   object        
 3   director          8807 non-null   object        
 4   cast              8807 non-null   object        
 5   country           8807 non-null   object        
 6   date_added        8797 non-null   datetime64[ns]
 7   release_year      8807 non-null   int64         
 8   rating            8807 non-null   category      
 9   listed_in         8807 non-null   object        
 10  description       8807 non-null   object        
 11  duration_minutes  6131 non-null   Int64         
 12  seasons           2676 non-null   Int64         
 13  year_added        8797 non-null   Int64         
 14  month_added       8797 n

In [29]:
clean.isna().sum()

show_id                0
type                   0
title                  0
director               0
cast                   0
country                0
date_added            10
release_year           0
rating                 0
listed_in              0
description            0
duration_minutes    2676
seasons             6131
year_added            10
month_added           10
dtype: int64

In [ ]:
clean.head()

## 7. Export the cleaned data

In [30]:
clean.to_csv("netflix_titles_cleaned.csv", index=False)
print(f"Saved {len(clean):,} rows to netflix_titles_cleaned.csv")

Saved 8,807 rows to netflix_titles_cleaned.csv


## Summary of changes

- Stripped whitespace and removed duplicate rows
- Recovered durations that had been typed into the `rating` column
- Filled missing `director`, `cast`, `country` and `rating` with `"Unknown"`
- Left missing `date_added` values empty instead of inventing dates
- Converted `duration` into `duration_minutes` (movies) and `seasons` (TV shows)
- Converted `date_added` to `datetime` and added `year_added` and `month_added`
- Converted `type` and `rating` to `category`
- Exported the result to `netflix_titles_cleaned.csv`